# Model 6 - Attribution Confidence (Leakage-Safe)

Goal: score confidence that a candidate post influenced a donation.

Leakage guards:
- Candidate posts must occur before the donation timestamp.
- Pair features exclude `post.donation_referrals`, `post.estimated_donation_value_php`, and other post-outcome donation fields.
- Time-based split by donation date.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.inspection import permutation_importance
import joblib

DATA_DIR = Path('.')
ARTIFACTS_DIR = DATA_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

donations = pd.read_csv(DATA_DIR / 'donations.csv', parse_dates=['donation_date'])
posts = pd.read_csv(DATA_DIR / 'social_media_posts.csv', parse_dates=['created_at'])

def time_split(df, time_col, frac=0.8):
    df = df.sort_values(time_col).copy()
    cut = int(len(df) * frac)
    return df.iloc[:cut].copy(), df.iloc[cut:].copy()

def build_preprocessor(X):
    num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat_cols = [c for c in X.columns if c not in num_cols]
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('pow', PowerTransformer(method='yeo-johnson', standardize=False)), ('sc', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ])


In [ ]:
base = donations[(donations['referral_post_id'].notna()) | (donations['channel_source'] == 'SocialMedia')].sort_values('donation_date').copy()
rows = []
for _, drow in base.iterrows():
    dtime = drow['donation_date']
    win = posts[(posts['created_at'] <= dtime) & (posts['created_at'] >= dtime - pd.Timedelta(days=7))].copy()
    if win.empty:
        continue
    win['hours_since_post'] = (dtime - win['created_at']).dt.total_seconds() / 3600
    win = win.sort_values(['hours_since_post', 'created_at']).head(20)

    for _, prow in win.iterrows():
        rows.append({
            'donation_id': drow['donation_id'],
            'donation_date': dtime,
            'post_id': prow['post_id'],
            'is_true_pair': int(pd.notna(drow['referral_post_id']) and int(drow['referral_post_id']) == int(prow['post_id'])),
            'hours_since_post': prow['hours_since_post'],
            'same_campaign_name': int(pd.notna(drow.get('campaign_name')) and pd.notna(prow.get('campaign_name')) and drow.get('campaign_name') == prow.get('campaign_name')),
            'donation_channel_source': drow.get('channel_source'),
            'platform': prow.get('platform'),
            'post_type': prow.get('post_type'),
            'media_type': prow.get('media_type'),
            'has_call_to_action': prow.get('has_call_to_action'),
            'is_boosted': prow.get('is_boosted'),
            'num_hashtags': prow.get('num_hashtags'),
            'caption_length': prow.get('caption_length'),
            'content_topic': prow.get('content_topic'),
            'sentiment_tone': prow.get('sentiment_tone'),
        })

m6 = pd.DataFrame(rows)
print('pair dataset:', m6.shape, 'positive rate:', m6['is_true_pair'].mean() if len(m6) else None)

if len(m6) > 100 and m6['is_true_pair'].nunique() > 1:
    train_df, test_df = time_split(m6, 'donation_date', 0.8)
    X_train = train_df.drop(columns=['is_true_pair', 'donation_id', 'post_id', 'donation_date'])
    y_train = train_df['is_true_pair']
    X_test = test_df.drop(columns=['is_true_pair', 'donation_id', 'post_id', 'donation_date'])
    y_test = test_df['is_true_pair']

    pre = build_preprocessor(X_train)
    predictive = Pipeline([('pre', pre), ('model', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42))])
    explanatory = Pipeline([('pre', pre), ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))])

    predictive.fit(X_train, y_train)
    explanatory.fit(X_train, y_train)

    prob = predictive.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    print({'roc_auc': roc_auc_score(y_test, prob), 'avg_precision': average_precision_score(y_test, prob), 'f1': f1_score(y_test, pred)})

    imp = permutation_importance(predictive, X_test, y_test, n_repeats=8, random_state=42)
    print(pd.DataFrame({'feature': X_test.columns, 'importance': imp.importances_mean}).sort_values('importance', ascending=False).head(10))

    joblib.dump(predictive, ARTIFACTS_DIR / 'model6_predictive.joblib')
    joblib.dump(explanatory, ARTIFACTS_DIR / 'model6_explanatory.joblib')
else:
    print('Insufficient pair signal. Increase lookback/candidate settings.')


In [ ]:
# Final business insights block (human-readable + actionable)

print('\n=== BUSINESS TAKEAWAYS: MODEL 6 (ATTRIBUTION CONFIDENCE) ===')

if 'predictive' not in globals():
    print('Model was not trained (likely due to limited pair data). Expand lookback/candidates first.')
else:
    scored = m6.copy()
    feature_cols = [c for c in scored.columns if c not in ['is_true_pair', 'donation_id', 'post_id', 'donation_date']]
    scored['attribution_confidence'] = predictive.predict_proba(scored[feature_cols])[:, 1]

    # Best candidate post per donation
    best_post = scored.sort_values(['donation_id', 'attribution_confidence'], ascending=[True, False]).groupby('donation_id', as_index=False).head(1)
    best_post['confidence_band'] = pd.cut(best_post['attribution_confidence'], bins=[-0.001, 0.35, 0.65, 1.0], labels=['Low', 'Medium', 'High'])

    print('Attribution confidence distribution (best-post-per-donation):')
    display(best_post['confidence_band'].value_counts(dropna=False).rename_axis('confidence_band').reset_index(name='count'))

    # Channel insight
    channel_insight = best_post.groupby(['platform', 'post_type'], dropna=False)['attribution_confidence'].mean().reset_index().sort_values('attribution_confidence', ascending=False)
    print('\nHighest-confidence platform/post-type combinations:')
    display(channel_insight.head(10))

    # Worklist for analysts
    print('\nTop attribution assignments (for review/dashboard):')
    display(best_post[['donation_id', 'post_id', 'attribution_confidence', 'confidence_band', 'platform', 'post_type', 'hours_since_post']].sort_values('attribution_confidence', ascending=False).head(20))

    print('\nActionable guidance:')
    print('- Use High-confidence assignments for reporting campaign effectiveness.')
    print('- Route Medium-confidence assignments to analyst review, and treat Low as unattributed.')
    print('- Prioritize platform/post-type combinations with consistently higher confidence for campaign planning.')
    print('- Attribution confidence is probabilistic support, not legal/causal proof.')